# 00 — Geração dos Splits de Avaliação

Gera os pares candidatos positivos e negativos para os conjuntos de **validação (2024)** e **teste (2025)**.
Esses pares são salvos uma única vez e reutilizados por todos os métodos (heurísticas, ML, GNN) para garantir uma comparação justa.

- **Positivos:** arestas que aparecem pela primeira vez em 2024 ou 2025 (novas colaborações).
- **Negativos:** pares de nós amostrados aleatoriamente que nunca colaboraram em nenhum ano do período 2018–2025.
- **Proporção:** 1 negativo por positivo (1:1).

Arquivos de saída salvos em `data/splits/`.

## 1. Imports e configuração

In [1]:
import random
import pandas as pd
import networkx as nx
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../data_collection").resolve()))
from construtor_grafo import subgrafo_novos, subgrafo_treino, ANO_VAL, ANO_TESTE

GRAPH_PATH = Path("../data/graphs/grafo_unico.graphml")
SPLITS_DIR = Path("../data/splits")
SEED = 42
SPLITS_DIR.mkdir(exist_ok=True)

## 2. Carregamento do grafo

O grafo já foi filtrado para a maior componente conexa (LCC) do subgrafo de treino durante sua construção.
Todos os nós presentes no arquivo são, portanto, nós válidos para amostragem.

In [2]:
G = nx.read_graphml(GRAPH_PATH)

print(f"Nós: {G.number_of_nodes()}")
print(f"Arestas totais (2018–2025): {G.number_of_edges()}")

Nós: 9154
Arestas totais (2018–2025): 63300


## 3. Pares positivos

Os positivos são as arestas que aparecem pela **primeira vez** em cada ano de avaliação.
A função `subgrafo_novos` usa o bitmask `anos_ativos` para identificar essas arestas em O(1).

In [3]:
val_pos  = [(min(u, v), max(u, v)) for u, v in subgrafo_novos(G, ANO_VAL).edges()]
test_pos = [(min(u, v), max(u, v)) for u, v in subgrafo_novos(G, ANO_TESTE).edges()]

print(f"Positivos — validação (2024): {len(val_pos)}")
print(f"Positivos — teste     (2025): {len(test_pos)}")

Positivos — validação (2024): 3142
Positivos — teste     (2025): 2423


## 4. Amostragem de pares negativos

Pares negativos são amostrados aleatoriamente entre nós do grafo que **nunca colaboraram** em nenhum ano (2018–2025).
Os negativos de validação e teste são amostrados sem reposição (ou seja, negativos de validação e teste não são sobrepostos).

In [4]:
random.seed(SEED)

all_nodes = list(G.nodes())
all_edges = {(min(u, v), max(u, v)) for u, v in G.edges()}

def sample_negatives(nodes, forbidden, n_samples):
    negatives = set()
    while len(negatives) < n_samples:
        u, v = random.sample(nodes, 2)
        pair = (min(u, v), max(u, v))
        if pair not in forbidden:
            negatives.add(pair)
    return list(negatives)

# negativos de validação
val_neg = sample_negatives(all_nodes, all_edges, len(val_pos))

# negativos de teste: excluem também os negativos já usados na validação
test_neg = sample_negatives(all_nodes, all_edges | set(val_neg), len(test_pos))

print(f"Negativos — validação (2024): {len(val_neg)}")
print(f"Negativos — teste     (2025): {len(test_neg)}")

Negativos — validação (2024): 3142
Negativos — teste     (2025): 2423


## 5. Salvamento dos splits

Cada arquivo CSV contém as colunas `node_u`, `node_v` e `label` (1 = positivo, 0 = negativo).
Esse formato é compatível com todos os métodos que serão avaliados.

In [5]:
def build_pairs_df(positives, negatives):
    rows = [(u, v, 1) for u, v in positives] + [(u, v, 0) for u, v in negatives]
    return pd.DataFrame(rows, columns=["node_u", "node_v", "label"])

val_df  = build_pairs_df(val_pos, val_neg)
test_df = build_pairs_df(test_pos, test_neg)

val_df.to_csv(SPLITS_DIR / "val_pairs.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test_pairs.csv", index=False)

print(f"val_pairs.csv  salvo  — {len(val_df)} pares ({val_df['label'].sum()} pos / {(val_df['label']==0).sum()} neg)")
print(f"test_pairs.csv salvo  — {len(test_df)} pares ({test_df['label'].sum()} pos / {(test_df['label']==0).sum()} neg)")

val_pairs.csv  salvo  — 6284 pares (3142 pos / 3142 neg)
test_pairs.csv salvo  — 4846 pares (2423 pos / 2423 neg)


## 6. Pares de treinamento (holdout mensagem/supervisão)

A partir do subgrafo de treino (2018–2023) aplicamos um **holdout 90/10** sobre as arestas, conforme `spec/message_supervision_split.md` (seção 2):

- **90% das arestas (msg)** permanecem em `G_train_msg` e servem como input/contexto: definem vizinhanças, features estruturais e topológicas, e o grafo de mensagem do GNN.
- **10% das arestas (supervisão)** saem do grafo e viram `train_pos`, os rótulos positivos vistos pelo modelo durante o treino.
- **Negativos:** pares de nós que nunca colaboraram em nenhum ano (2018–2025), excluindo também os negativos reservados para validação e teste. Quantidade igual a `len(train_pos)` para manter razão 1:1.

Sem essa separação, features como `degree_u`, `degree_v`, `preferential_attachment` e `jaccard` calculadas em `(u,v)` incluem a própria aresta-alvo, gerando uma distribuição de treino inconsistente com a de val/teste (onde as arestas-alvo nunca estão no grafo). No GNN, esse vazamento é catastrófico (trivial solution via message-passing). Detalhes e exemplo numérico na spec.


In [6]:
random.seed(SEED + 2)

G_train = subgrafo_treino(G)
todas_arestas_train = list(G_train.edges())
random.shuffle(todas_arestas_train)

n_total      = len(todas_arestas_train)
n_supervisao = int(round(n_total * 0.10))

train_pos_edges = todas_arestas_train[:n_supervisao]
msg_edges       = todas_arestas_train[n_supervisao:]

G_train_msg = G_train.edge_subgraph(msg_edges).copy()

train_pos = [(min(u, v), max(u, v)) for u, v in train_pos_edges]

nx.write_graphml(G_train_msg, "../data/graphs/grafo_treino_msg.graphml")

print(f"Arestas em G_train          : {n_total:,}")
print(f"Arestas em G_train_msg (90%): {len(msg_edges):,}")
print(f"Positivos de treino (10%)   : {len(train_pos):,}")
print(f"Nós em G_train_msg          : {G_train_msg.number_of_nodes():,}")

forbidden_treino = all_edges | set(val_neg) | set(test_neg)
train_neg = sample_negatives(all_nodes, forbidden_treino, len(train_pos))

train_df = build_pairs_df(train_pos, train_neg)
train_df.to_csv(SPLITS_DIR / "train_pairs.csv", index=False)

print(f"train_pairs.csv salvo — {len(train_df)} pares ({train_df['label'].sum()} pos / {(train_df['label']==0).sum()} neg)")


Arestas em G_train          : 57,735
Arestas em G_train_msg (90%): 51,961
Positivos de treino (10%)   : 5,774
Nós em G_train_msg          : 9,089
train_pairs.csv salvo — 11548 pares (5774 pos / 5774 neg)
